In [ ]:
"""
LIMPIEZA Y PREPROCESAMIENTO DE LOS DATOS
================================================================================
Fase 3 de CRISP-DM: Preparación de los Datos.


Aplica, en un solo bloque, las decisiones documentadas en el análisis
exploratorio:


  1. Eliminación de columnas no predictivas (rutas de imagen).
  2. Tratamiento de valores faltantes (eliminación e imputación).
  3. Eliminación de características redundantes por relaciones elevadas
     y dependencias matemáticas.
  4. Eliminación de orig_rango por su concentración marcada en los extremos.
  5. Conservación de las variables categóricas para su codificación
     posterior a la separación de los datos.
  6. Conservación de formato_entrada como información disponible antes
     de la recodificación.
  7. Conservación de LPIPS en su escala original.


Entrada:
    csv/dataset_final_orig.csv


Salida:
    csv/dataset_limpio.csv


El dataset de entrada contiene 29.991 filas y 33 columnas.
La salida se utilizará posteriormente para la separación de los datos
y el escalado de las variables durante la etapa de preparación para
el modelado.
"""


import pandas as pd
import numpy as np


CSV_PATH = "csv/dataset_final_orig.csv"  # <-- ajustar a la ruta real
df = pd.read_csv(CSV_PATH)
n_inicial = len(df)
print(f"Dataset cargado: {n_inicial} filas, {df.shape[1]} columnas")


# ------------------------------------------------------------------
# 1. ELIMINAR COLUMNAS NO PREDICTIVAS (rutas de archivo)
# ------------------------------------------------------------------
df = df.drop(columns=["img_original", "img_recodificada"])


# ------------------------------------------------------------------
# 2. TRATAMIENTO DE VALORES FALTANTES
# ------------------------------------------------------------------
# 2a. lpips: variable objetivo, no imputable -> eliminar filas
n_lpips_nulo = df["lpips"].isna().sum()
df = df.dropna(subset=["lpips"]).copy()


# 2b. orig_entropia_espectral: coincide exactamente con los 4 registros
#     degenerados (orig_hf = orig_lf = orig_hlfr = 0) -> eliminar filas
n_entesp_nulo = df["orig_entropia_espectral"].isna().sum()
df = df.dropna(subset=["orig_entropia_espectral"]).copy()


# 2c. orig_hue_var: un NaN no equivale a varianza circular = 0, sino a
#     ausencia de medicion valida (sin pixeles que superen el umbral de
#     saturacion). Imputar con 0 confundiria ambas condiciones. Se conserva
#     el NaN para imputarlo con la mediana calculada exclusivamente sobre
#     entrenamiento, despues de separar los datos. Se agrega una columna
#     indicadora binaria que preserva la distincion entre "medido" y
#     "no medible", permitiendo que el modelo aprenda de la ausencia si
#     resulta informativa.
n_hue_nulo = df["orig_hue_var"].isna().sum()
df["orig_hue_var_faltante"] = df["orig_hue_var"].isna().astype(int)


print(f"\nFilas eliminadas por lpips nulo: {n_lpips_nulo}")
print(f"Filas eliminadas por orig_entropia_espectral nulo: {n_entesp_nulo}")
print(f"Valores de orig_hue_var conservados para imputacion posterior: {n_hue_nulo}")
print("Se agrego la columna 'orig_hue_var_faltante' como indicador binario")
print(f"Filas restantes: {len(df)}")


# Verificar que orig_hue_var sea la unica variable con NaN.
columnas_con_nulos = df.columns[df.isna().any()].tolist()


if set(columnas_con_nulos) - {"orig_hue_var"}:
    detalle = df[columnas_con_nulos].isna().sum()
    raise ValueError(
        "Se encontraron valores faltantes inesperados:\n"
        f"{detalle[detalle > 0]}"
    )


# ------------------------------------------------------------------
# 3. ELIMINAR CARACTERISTICAS REDUNDANTES POR MULTICOLINEALIDAD (|r|>0.9)
# ------------------------------------------------------------------
columnas_redundantes = [
    "orig_var_lum",   # redundante con orig_std (r=0.9741)
    "orig_edges_s1",  # redundante con orig_grad_mean (r=0.9004)
    "orig_edges_s2",  # redundante con orig_edges_s3 y orig_grad_mean
    "orig_hlfr",      # derivada deterministicamente de orig_hf/orig_lf
]
df = df.drop(columns=columnas_redundantes)


# ------------------------------------------------------------------
# 4. ELIMINAR orig_rango (comportamiento casi binario por construccion)
# ------------------------------------------------------------------
df = df.drop(columns=["orig_rango"])


# ------------------------------------------------------------------
# 5. CONSERVAR LAS VARIABLES CATEGORICAS PARA CODIFICACION POSTERIOR
#    formato_entrada se conserva: es informacion conocida antes de la
#    recodificacion y potencialmente predictiva (p. ej. JPG->WebP y
#    PNG->WebP pueden producir perdidas perceptuales distintas).
#    calidad se mantiene numerica (orden natural: 10 < 30 < 50 < 70 < 90)
# ------------------------------------------------------------------
# La codificacion One-Hot se realizara despues de separar los datos,
# utilizando las categorias presentes en entrenamiento.


# ------------------------------------------------------------------
# 6. LPIPS se conserva sin transformacion (decision ya justificada
#    en el analisis exploratorio)
# ------------------------------------------------------------------


# ------------------------------------------------------------------
# RESUMEN Y GUARDADO
# ------------------------------------------------------------------
print("\n" + "=" * 70)
print("RESUMEN DE LA LIMPIEZA")
print("=" * 70)
print(f"Filas iniciales : {n_inicial}")
print(f"Filas finales    : {len(df)}")
print(f"Columnas finales : {df.shape[1]}")
print(f"\nColumnas resultantes:\n{df.columns.tolist()}")
print("\nValores faltantes por columna:")
print(df.isna().sum()[df.isna().sum() > 0])


df.to_csv("csv/dataset_limpio.csv", index=False)
print("\nGuardado: csv/dataset_limpio.csv")




In [ ]:
"""
SEPARACION DEL CONJUNTO DE DATOS Y NORMALIZACION
================================================================================
Continua sobre el dataset limpio (csv/dataset_limpio.csv, 29.985 filas).


1. Separa en train (70%) / validacion (15%) / prueba (15%), estratificado
   por 'calidad' para preservar la proporcion de los 5 niveles en cada
   subconjunto.
2. Normaliza (StandardScaler) unicamente las variables numericas continuas,
   ajustando el escalador solo sobre train para evitar fuga de informacion.
3. lpips se conserva en su escala original (no se normaliza, consistente
   con la decision tomada en el analisis exploratorio).
4. Las columnas binarias (dummies One-Hot y el indicador de faltante) no
   se normalizan.
"""


import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


CSV_PATH = "csv/dataset_limpio.csv"  # <-- ajustar a la ruta real
df = pd.read_csv(CSV_PATH)
print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")


TARGET = "lpips"


# ------------------------------------------------------------------
# 1. IDENTIFICAR COLUMNAS CATEGORICAS Y EL INDICADOR DE FALTANTE
# ------------------------------------------------------------------
cols_categoricas = ["metodo", "formato_salida", "formato_entrada"]
cols_indicador = ["orig_hue_var_faltante"]


# Verificar que las columnas necesarias existan
columnas_requeridas = [TARGET, "calidad", "orig_hue_var"] + cols_categoricas + cols_indicador
columnas_faltantes = [c for c in columnas_requeridas if c not in df.columns]


if columnas_faltantes:
    raise ValueError(f"Faltan columnas necesarias en el dataset: {columnas_faltantes}")


# ------------------------------------------------------------------
# 2. SEPARACION ESTRATIFICADA 70 / 15 / 15
# ------------------------------------------------------------------
X = df.drop(columns=[TARGET]).copy()
y = df[TARGET].copy()
estrato = df["calidad"]


# Primero: 70% train vs 30% restante (validacion + prueba)
X_train, X_temp, y_train, y_temp, estrato_train, estrato_temp = train_test_split(
    X, y, estrato,
    test_size=0.30,
    stratify=estrato,
    random_state=42,
)


# Segundo: dividir el 30% restante en 15% validacion y 15% prueba
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=estrato_temp,
    random_state=42,
)


X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()


print(f"\nTrain: {len(X_train)} filas ({len(X_train)/len(df)*100:.1f}%)")
print(f"Validacion: {len(X_val)} filas ({len(X_val)/len(df)*100:.1f}%)")
print(f"Prueba: {len(X_test)} filas ({len(X_test)/len(df)*100:.1f}%)")


# Verificacion de balance de 'calidad' en cada subconjunto
print("\nDistribucion de 'calidad' por subconjunto (%):")
for nombre, subset in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    print(f"  {nombre}:")
    print(
        (subset["calidad"].value_counts(normalize=True).sort_index() * 100)
        .round(2)
        .to_string()
    )


# ------------------------------------------------------------------
# 3. TRATAMIENTO DE VALORES FALTANTES (FIT SOLO SOBRE TRAIN)
# ------------------------------------------------------------------
# Imputar orig_hue_var con la mediana calculada únicamente sobre train
mediana_hue = X_train["orig_hue_var"].median()


if pd.isna(mediana_hue):
    raise ValueError(
        "No se pudo calcular la mediana de orig_hue_var en train."
    )


X_train.loc[:, "orig_hue_var"] = X_train["orig_hue_var"].fillna(mediana_hue)
X_val.loc[:, "orig_hue_var"] = X_val["orig_hue_var"].fillna(mediana_hue)
X_test.loc[:, "orig_hue_var"] = X_test["orig_hue_var"].fillna(mediana_hue)


print(f"\nMediana de orig_hue_var calculada sobre train: {mediana_hue:.5f}")


# Verificar que no queden valores faltantes en las caracteristicas
for nombre, subset in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    columnas_nulas = subset.columns[subset.isna().any()].tolist()
    if columnas_nulas:
        raise ValueError(
            f"Hay valores faltantes en {nombre}: {columnas_nulas}"
        )


# ------------------------------------------------------------------
# 4. CODIFICACION ONE-HOT DESPUES DE LA SEPARACION
# ------------------------------------------------------------------
# Las categorias se obtienen de cada subconjunto; posteriormente se
# alinean validacion y prueba con las columnas generadas en train.
X_train = pd.get_dummies(
    X_train,
    columns=cols_categoricas,
    dtype=int
)


X_val = pd.get_dummies(
    X_val,
    columns=cols_categoricas,
    dtype=int
)


X_test = pd.get_dummies(
    X_test,
    columns=cols_categoricas,
    dtype=int
)


# Alinear validacion y prueba con las columnas generadas en train
columnas_train = X_train.columns.tolist()


X_val = X_val.reindex(columns=columnas_train, fill_value=0)
X_test = X_test.reindex(columns=columnas_train, fill_value=0)


# Verificar que no queden columnas de texto
for nombre, subset in [("Train", X_train), ("Val", X_val), ("Test", X_test)]:
    columnas_texto = subset.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()


    if columnas_texto:
        raise ValueError(
            f"Quedaron columnas de texto sin codificar en {nombre}: "
            f"{columnas_texto}"
        )


# ------------------------------------------------------------------
# 5. IDENTIFICAR COLUMNAS A NORMALIZAR VS. COLUMNAS BINARIAS
# ------------------------------------------------------------------
prefijos_dummy = ("metodo_", "formato_salida_", "formato_entrada_")


cols_binarias = [
    c for c in X_train.columns
    if c.startswith(prefijos_dummy)
]
cols_binarias.extend(cols_indicador)


# Eliminar duplicados conservando el orden
cols_binarias = list(dict.fromkeys(cols_binarias))


# Verificar que las columnas binarias existan
cols_binarias = [c for c in cols_binarias if c in X_train.columns]


# Identificar variables continuas a normalizar
cols_continuas = [
    c for c in X_train.columns
    if c not in cols_binarias
]


# Verificar que todas las columnas continuas sean numericas
columnas_no_numericas = X_train[cols_continuas].select_dtypes(
    exclude=[np.number]
).columns.tolist()


if columnas_no_numericas:
    raise ValueError(
        f"Hay columnas no numericas entre las variables continuas: "
        f"{columnas_no_numericas}"
    )


print(f"\nColumnas continuas a normalizar ({len(cols_continuas)}): {cols_continuas}")
print(f"\nColumnas binarias (sin normalizar) ({len(cols_binarias)}): {cols_binarias}")


# ------------------------------------------------------------------
# 6. NORMALIZACION (FIT SOLO SOBRE TRAIN)
# ------------------------------------------------------------------
scaler = StandardScaler()


X_train.loc[:, cols_continuas] = scaler.fit_transform(
    X_train[cols_continuas]
)
X_val.loc[:, cols_continuas] = scaler.transform(
    X_val[cols_continuas]
)
X_test.loc[:, cols_continuas] = scaler.transform(
    X_test[cols_continuas]
)


print("\nNormalizacion aplicada (media=0, desviacion=1 sobre train).")
print("Verificacion sobre train (deberia ser ~0 y ~1):")
print(X_train[cols_continuas].mean().round(3).head())
print(X_train[cols_continuas].std().round(3).head())


# Verificar dimensiones y correspondencia de columnas
assert X_train.columns.tolist() == X_val.columns.tolist()
assert X_train.columns.tolist() == X_test.columns.tolist()


print("\nDimensiones finales:")
print(f"X_train: {X_train.shape}")
print(f"X_val:   {X_val.shape}")
print(f"X_test:  {X_test.shape}")


# ------------------------------------------------------------------
# 7. GUARDAR LOS CONJUNTOS RESULTANTES
# ------------------------------------------------------------------
X_train.to_csv("csv/X_train.csv", index=False)
X_val.to_csv("csv/X_val.csv", index=False)
X_test.to_csv("csv/X_test.csv", index=False)
y_train.to_csv("csv/y_train.csv", index=False)
y_val.to_csv("csv/y_val.csv", index=False)
y_test.to_csv("csv/y_test.csv", index=False)


print(
    "\nArchivos guardados: X_train.csv, X_val.csv, X_test.csv, "
    "y_train.csv, y_val.csv, y_test.csv"
)